In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

gold_df = spark.read.table("workspace.default.gold_overdose")

print(f"Total rows: {gold_df.count()}")
display(gold_df)

Total rows: 45813


State,Indicator,Date,Death_Count,Percent_Complete,Percent_Pending_Investigation,State_Name,Predicted_Value,year,month,quarter,lag_1,lag_3,roll_mean_3,roll_std_3
AK,Cocaine (T40.5),2016-07-01,13.0,100.0,0.0,Alaska,13.0,2016,7,3,11.0,11.0,11.666666666666666,1.1547005383792517
AK,Cocaine (T40.5),2016-08-01,12.0,100.0,0.02338634238,Alaska,12.0,2016,8,3,13.0,11.0,12.0,1.0
AK,Cocaine (T40.5),2016-09-01,11.0,100.0,0.0469924812,Alaska,11.0,2016,9,3,12.0,11.0,12.0,1.0
AK,Cocaine (T40.5),2016-10-01,13.0,100.0,0.06991377301,Alaska,13.0,2016,10,4,11.0,13.0,12.0,1.0
AK,Cocaine (T40.5),2016-11-01,15.0,100.0,0.06994637445,Alaska,15.0,2016,11,4,13.0,12.0,13.0,2.0
AK,Cocaine (T40.5),2016-12-01,15.0,100.0,0.06888633754,Alaska,15.0,2016,12,4,15.0,11.0,14.333333333333334,1.1547005383792517
AK,Cocaine (T40.5),2017-01-01,15.0,100.0,0.06858710562,Alaska,15.0,2017,1,1,15.0,13.0,15.0,0.0
AK,Cocaine (T40.5),2017-02-01,18.0,100.0,0.06901311249,Alaska,18.0,2017,2,1,15.0,15.0,16.0,1.7320508075688772
AK,Cocaine (T40.5),2017-03-01,19.0,100.0,0.06882312457,Alaska,19.0,2017,3,1,18.0,15.0,17.333333333333332,2.0816659994661326
AK,Cocaine (T40.5),2017-04-01,16.0,100.0,0.06936416185,Alaska,16.0,2017,4,2,19.0,15.0,17.666666666666668,1.5275252316519465


In [0]:
# read from gold table
from pyspark.sql.functions import col, year

gold_df = spark.read.table("workspace.default.gold_overdose")

# temporal split matching phase 2 70/15/15
train_df = gold_df.filter(col("year") <= 2022)
val_df = gold_df.filter(col("year") == 2023)
test_df = gold_df.filter(col("year") >= 2024)

print(f"Train rows: {train_df.count()}")
print(f"Val rows: {val_df.count()}")
print(f"Test rows: {test_df.count()}")

Train rows: 31371
Val rows: 5378
Test rows: 9064


In [0]:

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

# step 1: convert State and Indicator strings to numeric indices
state_indexer = StringIndexer(inputCol="State", outputCol="State_idx", handleInvalid="keep")
indicator_indexer = StringIndexer(inputCol="Indicator", outputCol="Indicator_idx", handleInvalid="keep")

# step 2: one hot encode the indexed categorical columns
state_encoder = OneHotEncoder(inputCol="State_idx", outputCol="State_vec")
indicator_encoder = OneHotEncoder(inputCol="Indicator_idx", outputCol="Indicator_vec")

# step 3: assemble all features into one vector
numeric_features = [
    "Percent_Complete", "Percent_Pending_Investigation",
    "year", "month", "quarter",
    "lag_1", "lag_3", "roll_mean_3", "roll_std_3"
]

assembler = VectorAssembler(
    inputCols=["State_vec", "Indicator_vec"] + numeric_features,
    outputCol="features_raw",
    handleInvalid="keep"
)

# step 4: scale features 
scaler = StandardScaler(inputCol="features_raw", outputCol="features")

print("Feature pipeline defined")

Feature pipeline defined


In [0]:
# Model 1: Ridge Regression

from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# build the full pipeline
ridge = LinearRegression(
    featuresCol="features",
    labelCol="Death_Count",
    regParam=1.0,
    elasticNetParam=0.0
)

ridge_pipeline = Pipeline(stages=[
    state_indexer, indicator_indexer,
    state_encoder, indicator_encoder,
    assembler, scaler, ridge
])

# train
ridge_model = ridge_pipeline.fit(train_df)

# evaluate on val and test
evaluator_rmse = RegressionEvaluator(labelCol="Death_Count", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="Death_Count", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="Death_Count", predictionCol="prediction", metricName="r2")

val_preds = ridge_model.transform(val_df)
test_preds = ridge_model.transform(test_df)

print("Ridge Regression Results:")
print(f"  Val | RMSE: {evaluator_rmse.evaluate(val_preds):.3f}, MAE: {evaluator_mae.evaluate(val_preds):.3f}, R2: {evaluator_r2.evaluate(val_preds):.3f}")
print(f"  Test | RMSE: {evaluator_rmse.evaluate(test_preds):.3f}, MAE: {evaluator_mae.evaluate(test_preds):.3f}, R2: {evaluator_r2.evaluate(test_preds):.3f}")

Ridge Regression Results:
  Val | RMSE: 140.920, MAE: 29.978, R2: 1.000
  Test | RMSE: 452.688, MAE: 86.957, R2: 0.995


In [0]:
# Model 2: Random Forest

from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features_raw", 
    labelCol="Death_Count",
    numTrees=300,
    maxDepth=10,
    seed=42
)

rf_pipeline = Pipeline(stages=[
    state_indexer, indicator_indexer,
    state_encoder, indicator_encoder,
    assembler, rf
])

# train
rf_model = rf_pipeline.fit(train_df)

# evaluate
val_preds_rf = rf_model.transform(val_df)
test_preds_rf = rf_model.transform(test_df)

print("Random Forest Results:")
print(f"  Val | RMSE: {evaluator_rmse.evaluate(val_preds_rf):.3f}, MAE: {evaluator_mae.evaluate(val_preds_rf):.3f}, R2: {evaluator_r2.evaluate(val_preds_rf):.3f}")
print(f"  Test | RMSE: {evaluator_rmse.evaluate(test_preds_rf):.3f}, MAE: {evaluator_mae.evaluate(test_preds_rf):.3f}, R2: {evaluator_r2.evaluate(test_preds_rf):.3f}")

Random Forest Results:
  Val | RMSE: 1677.085, MAE: 262.247, R2: 0.963
  Test | RMSE: 1995.201, MAE: 338.269, R2: 0.908


In [0]:
# Model 3: Gradient Boosting

from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features_raw",  
    labelCol="Death_Count",
    maxIter=100,
    maxDepth=8,
    stepSize=0.05,
    seed=42
)

gbt_pipeline = Pipeline(stages=[
    state_indexer, indicator_indexer,
    state_encoder, indicator_encoder,
    assembler, gbt
])

# train
gbt_model = gbt_pipeline.fit(train_df)

# evaluate
val_preds_gbt = gbt_model.transform(val_df)
test_preds_gbt = gbt_model.transform(test_df)

print("Gradient Boosting Results:")
print(f"  Val | RMSE: {evaluator_rmse.evaluate(val_preds_gbt):.3f}, MAE: {evaluator_mae.evaluate(val_preds_gbt):.3f}, R2: {evaluator_r2.evaluate(val_preds_gbt):.3f}")
print(f"  Test | RMSE: {evaluator_rmse.evaluate(test_preds_gbt):.3f}, MAE: {evaluator_mae.evaluate(test_preds_gbt):.3f}, R2: {evaluator_r2.evaluate(test_preds_gbt):.3f}")

Gradient Boosting Results:
  Val | RMSE: 577.580, MAE: 114.924, R2: 0.996
  Test | RMSE: 2165.717, MAE: 297.676, R2: 0.891


In [0]:


import pandas as pd

comparison = pd.DataFrame({
    'Model': ['Ridge Regression', 'Random Forest', 'Gradient Boosting'],
    
    # Spark MLlib results
    'Spark Val RMSE': [140.920, 1677.085, 577.580],
    'Spark Test RMSE': [452.688, 1995.201, 2165.717],
    'Spark Test R2': [0.995, 0.908, 0.891],
    
    # Phase 2 sklearn results from report
    'Phase2 Val RMSE': [173.249, 328.182, 383.930],
    'Phase2 Test RMSE': [339.566, 451.064, 446.287],
    'Phase2 Test R2': [0.997, 0.995, 0.995],
})

print("Model Comparison: Spark MLlib vs Phase 2 sklearn")
display(comparison)

Model Comparison: Spark MLlib vs Phase 2 sklearn


Model,Spark Val RMSE,Spark Test RMSE,Spark Test R2,Phase2 Val RMSE,Phase2 Test RMSE,Phase2 Test R2
Ridge Regression,140.92,452.688,0.995,173.249,339.566,0.997
Random Forest,1677.085,1995.201,0.908,328.182,451.064,0.995
Gradient Boosting,577.58,2165.717,0.891,383.93,446.287,0.995


In [0]:
# save all 3 trained models
ridge_model.write().overwrite().save("/Volumes/workspace/default/dic_project/models/ridge_model")
rf_model.write().overwrite().save("/Volumes/workspace/default/dic_project/models/rf_model")
gbt_model.write().overwrite().save("/Volumes/workspace/default/dic_project/models/gbt_model")

print("All models saved successfully")

All models saved successfully
